In [ ]:
import os
import rasterio
from rasterio.warp import calculate_default_transform, Resampling, reproject
from rasterio.vrt import WarpedVRT
from tqdm import tqdm

SRC_4326 = r"E:\DownloadData\soilgrids"
OUT_32648 = r"E:\DownloadData\co2_ban_do\output_32648_soilgrids"
os.makedirs(OUT_32648, exist_ok=True)

TARGET_CRS = "EPSG:32648"

files = [f for f in os.listdir(SRC_4326) if f.endswith(".tif")]

for fname in tqdm(files, desc="Reproject 4326 → 32648"):
    src_path = os.path.join(SRC_4326, fname)
    out_path = os.path.join(OUT_32648, fname.replace(".tif", "_32648.tif"))

    with rasterio.open(src_path) as src:

        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )

        profile = src.profile.copy()
        profile.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
        })

        with rasterio.open(out_path, "w", **profile) as dst:
            for b in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, b),
                    destination=rasterio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.nearest   # GIỮ giá trị
                )


Reproject 4326 → 32648: 100%|██████████| 6/6 [00:15<00:00,  2.64s/it]


In [6]:
import os
import rasterio
import numpy as np
from tqdm import tqdm
from rasterio.warp import calculate_default_transform, reproject, Resampling

SRC_ROOT = r"E:\DownloadData\gee\modis_climate_tif"
OUT_ROOT = r"E:\DownloadData\co2_ban_do\output_32648"
TARGET_CRS = "EPSG:32648"

os.makedirs(OUT_ROOT, exist_ok=True)

def convert_file(src_path, dst_path):
    with rasterio.open(src_path) as src:

        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds
        )

        profile = src.profile.copy()
        profile.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
            "nodata": np.nan,        # QUAN TRỌNG
            "dtype": "float32"       # ERA/MODIS đều float
        })

        with rasterio.open(dst_path, "w", **profile) as dst:

            # giữ tên band
            if src.descriptions:
                for i, desc in enumerate(src.descriptions):
                    dst.set_band_description(i+1, desc)

            for b in range(1, src.count + 1):

                arr = np.zeros((height, width), dtype="float32")

                reproject(
                    source=rasterio.band(src, b),
                    destination=arr,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.bilinear   # THAY nearest !
                )

                dst.write(arr, b)


# DUYỆT dataset
for dataset in os.listdir(SRC_ROOT):

    dataset_path = os.path.join(SRC_ROOT, dataset)
    if not os.path.isdir(dataset_path):
        continue

    print("\n===============================")
    print("📌 Dataset:", dataset)
    print("===============================")

    for year_folder in os.listdir(dataset_path):

        year_path = os.path.join(dataset_path, year_folder)
        if not os.path.isdir(year_path):
            continue

        out_year_path = os.path.join(OUT_ROOT, dataset, year_folder)
        os.makedirs(out_year_path, exist_ok=True)

        tifs = [f for f in os.listdir(year_path) if f.endswith(".tif")]

        for fname in tqdm(tifs, desc=f"{dataset} {year_folder}"):

            src_file = os.path.join(year_path, fname)
            out_file = os.path.join(
                out_year_path,
                fname.replace(".tif", "_32648.tif")
            )

            convert_file(src_file, out_file)

print("\n🎉 DONE — Bước 1 chính xác, không mất band, không mất tên band, không sai giá trị!")



🎉 DONE — Bước 1 chính xác, không mất band, không mất tên band, không sai giá trị!
